# Отбор agr_id по категориям совпадения monthly commission (Q1 2026)

Цель:
- выбрать по 1 `agr_id` в каждой из 3 категорий за Январь/Февраль/Март;
- после отбора получить для выбранных `agr_id` объединенную информацию из:
  - `ods_alpha.scd1_agreements`
  - `ods_alpha.scd1_agr_terms`
  - `ods.scd1_z_r2_ip_merchants`

In [ ]:
import re

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)

In [ ]:
excel_sources = [
    {'report_month': '2026-01-01', 'path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

months_q1 = ['2026-01', '2026-02', '2026-03']
sample_size_per_category = 1

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

## 1) Загрузка Excel Q1

In [ ]:
def normalize_colname(value):
    s = str(value).lower().replace('\n', ' ').replace('\r', ' ').replace('\xa0', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    s = s.replace('₽', 'руб').replace('%', 'pct')
    s = re.sub(r'[^a-zа-я0-9]+', '', s)
    return s


def pick_column(columns, aliases):
    norm_map = {normalize_colname(c): c for c in columns}
    for alias in aliases:
        key = normalize_colname(alias)
        if key in norm_map:
            return norm_map[key]
    return None


def normalize_agr_id(value):
    if pd.isna(value):
        return None
    s = str(value).strip()
    if s.lower() in {'', 'nan', 'none'}:
        return None
    return s


frames = []
for src in excel_sources:
    raw = pd.read_excel(src['path'], header=src['header'])

    agr_col = pick_column(raw.columns, ['ID договора', 'agr_id'])
    cm_col = pick_column(raw.columns, ['Комиссия \n(₽ в месяц)', 'Комиссия (₽ в месяц)'])

    if agr_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка agr_id")
    if cm_col is None:
        raise ValueError(f"В файле {src['path']} не найдена колонка monthly commission")

    df = pd.DataFrame({
        'agr_id': raw[agr_col].apply(normalize_agr_id),
        'commission_monthly_excel': pd.to_numeric(raw[cm_col], errors='coerce'),
    })
    df['report_month'] = pd.to_datetime(src['report_month'])
    frames.append(df)

excel_df = pd.concat(frames, ignore_index=True)
excel_df = excel_df[excel_df['agr_id'].notna()].copy()
excel_df['report_month'] = pd.to_datetime(excel_df['report_month'], errors='coerce')
excel_df['report_month_str'] = excel_df['report_month'].dt.strftime('%Y-%m')
excel_df = excel_df[excel_df['report_month_str'].isin(months_q1)].copy()

print(f'Excel rows Q1: {len(excel_df):,}')
print(excel_df.groupby('report_month_str').size())
excel_df.head(5)

## 2) Подтяжка Озера: c_summa и tariff_name по agr_id

In [ ]:
agr_values = sorted(excel_df['agr_id'].astype(str).unique().tolist())
agr_in = ', '.join([f"'{x}'" for x in agr_values]) if agr_values else "''"

sql_lake = f"""
select
  cast(m.id as string) as agr_id,
  cast(max(tp.c_name) as string) as tariff_name,
  max(cast(tf.c_summa as decimal(18,2))) as commission_monthly_lake,
  cast(max(m.c_cl_org) as string) as c_cl_org,
  cast(max(m.c_tariff_plan) as string) as c_tariff_plan,
  cast(max(m.c_depart) as string) as c_depart
from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_fix tf
  on cast(tf.id as string) = cast(tt.c_tariff as string)
where cast(m.id as string) in ({agr_in})
group by cast(m.id as string)
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    lake_df = imp.fetch(sql_lake)

if lake_df is None:
    lake_df = pd.DataFrame(columns=[
        'agr_id', 'tariff_name', 'commission_monthly_lake',
        'c_cl_org', 'c_tariff_plan', 'c_depart'
    ])

if not lake_df.empty:
    lake_df['agr_id'] = lake_df['agr_id'].astype(str)
    lake_df['tariff_name'] = lake_df['tariff_name'].astype('string')
    lake_df['commission_monthly_lake'] = pd.to_numeric(lake_df['commission_monthly_lake'], errors='coerce')

print(f'Lake rows: {len(lake_df):,}')
lake_df.head(5)

## 3) Категоризация agr_id (по 3 месяцам, по 1 штуке)

In [ ]:
merged_df = excel_df.merge(lake_df, on='agr_id', how='left')

# Спец-правило из предыдущей логики: NaN в lake и 0.0 в Excel считаем совпадением
nan_lake_zero_excel_match = (
    merged_df['commission_monthly_lake'].isna()
    & merged_df['commission_monthly_excel'].notna()
    & (merged_df['commission_monthly_excel'] == 0)
)

merged_df['is_diff_strict'] = (
    (merged_df['commission_monthly_excel'].isna() & merged_df['commission_monthly_lake'].notna())
    | (merged_df['commission_monthly_excel'].notna() & merged_df['commission_monthly_lake'].isna() & (~nan_lake_zero_excel_match))
    | (
        merged_df['commission_monthly_excel'].notna()
        & merged_df['commission_monthly_lake'].notna()
        & (merged_df['commission_monthly_excel'] != merged_df['commission_monthly_lake'])
    )
)

# Дедуп на уровень agr_id+month
pm = merged_df[[
    'agr_id', 'report_month_str', 'commission_monthly_lake', 'commission_monthly_excel',
    'is_diff_strict', 'tariff_name', 'c_cl_org', 'c_tariff_plan', 'c_depart'
]].copy()
pm = pm.sort_values(['agr_id', 'report_month_str']).drop_duplicates(['agr_id', 'report_month_str'], keep='first')

pm['lake_gt_0'] = pm['commission_monthly_lake'].notna() & (pm['commission_monthly_lake'] > 0)
pm['excel_eq_0'] = pm['commission_monthly_excel'].notna() & (pm['commission_monthly_excel'] == 0)
pm['excel_gt_0'] = pm['commission_monthly_excel'].notna() & (pm['commission_monthly_excel'] > 0)
pm['is_match_month'] = ~pm['is_diff_strict']
pm['is_mismatch_month'] = pm['is_diff_strict']

agg = (
    pm.groupby('agr_id', as_index=False)
    .agg(
        months_cnt=('report_month_str', 'nunique'),
        match_months=('is_match_month', 'sum'),
        mismatch_months=('is_mismatch_month', 'sum'),
        lake_gt_0_months=('lake_gt_0', 'sum'),
        excel_eq_0_months=('excel_eq_0', 'sum'),
        excel_gt_0_months=('excel_gt_0', 'sum'),
    )
)
agg = agg[agg['months_cnt'] == 3].copy()

# Категория 1: совпадает 3/3 и в lake c_summa > 0 (3/3)
cat1_all = agg.loc[
    (agg['match_months'] == 3) & (agg['lake_gt_0_months'] == 3),
    'agr_id'
].sort_values().tolist()

# Категория 2: mismatch 3/3, lake >0 3/3, excel =0 3/3
cat2_all = agg.loc[
    (agg['mismatch_months'] == 3)
    & (agg['lake_gt_0_months'] == 3)
    & (agg['excel_eq_0_months'] == 3),
    'agr_id'
].sort_values().tolist()

# Категория 3: mismatch 3/3, lake >0 3/3, excel >0 3/3
cat3_all = agg.loc[
    (agg['mismatch_months'] == 3)
    & (agg['lake_gt_0_months'] == 3)
    & (agg['excel_gt_0_months'] == 3),
    'agr_id'
].sort_values().tolist()

cat1_selected = cat1_all[:sample_size_per_category]
cat2_selected = cat2_all[:sample_size_per_category]
cat3_selected = cat3_all[:sample_size_per_category]

print('Всего найдено по категориям (до лимита 1):')
print(f'  cat1 (match 3/3, lake>0): {len(cat1_all):,}')
print(f'  cat2 (mismatch 3/3, lake>0, excel=0): {len(cat2_all):,}')
print(f'  cat3 (mismatch 3/3, lake>0, excel>0): {len(cat3_all):,}')

print('\nВыбранные agr_id (до 1 на категорию):')
print('cat1_selected =', cat1_selected)
print('cat2_selected =', cat2_selected)
print('cat3_selected =', cat3_selected)

selected_agr_ids = sorted(set(cat1_selected + cat2_selected + cat3_selected))
print(f'\nВсего уникальных выбранных agr_id: {len(selected_agr_ids)}')

# Показываем помесячную детализацию по выбранным agr_id
selected_monthly_df = pm[pm['agr_id'].isin(selected_agr_ids)].copy()
selected_monthly_df = selected_monthly_df.sort_values(['agr_id', 'report_month_str'])
print('\nПомесячная детализация выбранных agr_id:')
selected_monthly_df

## 4) Agreements + agr_terms + merchants в одном DataFrame для выбранных agr_id

In [ ]:
if not selected_agr_ids:
    raise ValueError('Не найдено agr_id для категорий. Ослабь условия или проверь исходные данные.')

sel_in = ', '.join([f"'{x}'" for x in selected_agr_ids])

sql_joined = f"""
with base_agreements as (
    select
        cast(a.abs_agr_id as string) as agr_id,
        cast(a.n_agr as string) as n_agr,
        cast(a.n_cmp_client as string) as n_cmp_client,
        cast(a.acq_class as string) as acq_class,
        cast(a.cf_agr_financial as string) as cf_agr_financial,
        cast(a.d_valid_from as timestamp) as d_valid_from,
        cast(a.d_valid_to as timestamp) as d_valid_to
    from ods_alpha.scd1_agreements a
    where cast(a.abs_agr_id as string) in ({sel_in})
),
agr_terms_one as (
    select
        cast(t.n_agr as string) as n_agr,
        cast(max(t.c_nmrc) as string) as c_nmrc,
        cast(max(t.d_valid_from) as timestamp) as agr_terms_d_start,
        cast(max(t.d_valid_to) as timestamp) as agr_terms_d_end
    from ods_alpha.scd1_agr_terms t
    where cast(t.n_agr as string) in (select n_agr from base_agreements)
    group by cast(t.n_agr as string)
),
merchants_one as (
    select
        cast(m.id as string) as agr_id,
        cast(m.c_cl_org as string) as c_cl_org,
        cast(m.c_tariff_plan as string) as c_tariff_plan,
        cast(m.c_depart as string) as c_depart
    from ods.scd1_z_r2_ip_merchants m
    where cast(m.id as string) in ({sel_in})
)
select
    ba.agr_id,
    ba.n_agr,
    ba.n_cmp_client,
    ba.acq_class,
    ba.cf_agr_financial,
    ba.d_valid_from,
    ba.d_valid_to,
    agt.c_nmrc,
    agt.agr_terms_d_start,
    agt.agr_terms_d_end,
    mo.c_cl_org,
    mo.c_tariff_plan,
    mo.c_depart
from base_agreements ba
left join agr_terms_one agt
  on agt.n_agr = ba.n_agr
left join merchants_one mo
  on mo.agr_id = ba.agr_id
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    joined_df = imp.fetch(sql_joined)

if joined_df is None:
    joined_df = pd.DataFrame()

if not joined_df.empty:
    joined_df['agr_id'] = joined_df['agr_id'].astype(str)

# Категорию добавляем на стороне pandas
agr_to_category = {x: 'cat1_match3m_lake_gt0' for x in cat1_selected}
agr_to_category.update({x: 'cat2_mismatch3m_lake_gt0_excel0' for x in cat2_selected})
agr_to_category.update({x: 'cat3_mismatch3m_lake_gt0_excel_gt0' for x in cat3_selected})

joined_df['category'] = joined_df['agr_id'].map(agr_to_category)

# Добавим агрегированную помесячную справку (Excel/Lake) для удобства ручной проверки
monthly_pivot = selected_monthly_df.copy()
monthly_pivot['month_pair'] = (
    monthly_pivot['report_month_str']
    + ': lake=' + monthly_pivot['commission_monthly_lake'].astype(str)
    + ', excel=' + monthly_pivot['commission_monthly_excel'].astype(str)
)
monthly_summary = (
    monthly_pivot.groupby('agr_id', as_index=False)['month_pair']
    .agg(lambda s: ' | '.join(sorted(s.tolist())))
    .rename(columns={'month_pair': 'monthly_check'})
)

final_selected_df = joined_df.merge(monthly_summary, on='agr_id', how='left')
final_selected_df = final_selected_df.sort_values(['category', 'agr_id']).reset_index(drop=True)

print(f'Rows in final_selected_df: {len(final_selected_df):,}')
print('Rows by category:')
print(final_selected_df['category'].value_counts(dropna=False))

final_selected_df

## 5) Единый DataFrame с подтвержденными тарифными данными по выбранным agr_id

Собираем только подтвержденную часть цепочки:
- `ods.scd1_z_r2_ip_merchants`
- `ods.scd1_z_r2_tariff_plan`
- `ods.scd1_z_r2_tariff_tune`
- `ods.scd1_z_r2_tariff_fix`
- `ods.scd1_z_r2_tariff_calc`

`ods.scd1_z_r2_tariff_comiss` и `ods.scd1_z_r2_tariffs` пока не используем (до фикса merge-логики).

In [ ]:
if not selected_agr_ids:
    raise ValueError('Нет выбранных agr_id для секции тарифов.')

sel_in = ', '.join([f"'{x}'" for x in selected_agr_ids])

# Только проверенные атрибуты тарифной цепочки
sql_tariff_full = f"""
select
    cast(m.id as string) as agr_id,
    cast(m.c_tariff_plan as string) as c_tariff_plan,

    cast(tp.id as string) as tariff_plan_id,
    cast(tp.c_name as string) as tariff_plan_name,
    cast(tp.c_code as string) as tariff_plan_code,

    cast(tt.c_tariff as string) as tariff_id,
    cast(tt.c_rule as string) as tariff_rule,
    cast(tt.c_vid_comiss as string) as vid_comiss_id,

    cast(tf.id as string) as tariff_fix_id,
    cast(tf.c_summa as decimal(18,2)) as fix_summa,

    cast(tc.id as string) as tariff_calc_id,
    cast(tc.c_name as string) as tariff_calc_name

from ods.scd1_z_r2_ip_merchants m
left join ods.scd1_z_r2_tariff_plan tp
  on cast(tp.id as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_tune tt
  on cast(tt.c_tariff_plan as string) = cast(m.c_tariff_plan as string)
left join ods.scd1_z_r2_tariff_fix tf
  on cast(tf.id as string) = cast(tt.c_tariff as string)
left join ods.scd1_z_r2_tariff_calc tc
  on cast(tc.id as string) = cast(tt.c_tariff as string)
where cast(m.id as string) in ({sel_in})
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    tariff_full_df = imp.fetch(sql_tariff_full)

if tariff_full_df is None:
    tariff_full_df = pd.DataFrame()

if not tariff_full_df.empty:
    tariff_full_df['agr_id'] = tariff_full_df['agr_id'].astype(str)

# Добавим категорию и помесячную проверку из уже собранных результатов
tariff_full_df = tariff_full_df.merge(
    final_selected_df[['agr_id', 'category', 'monthly_check']].drop_duplicates(),
    on='agr_id',
    how='left'
)

tariff_full_df = tariff_full_df.sort_values(['category', 'agr_id']).reset_index(drop=True)

print(f'Rows in tariff_full_df: {len(tariff_full_df):,}')
print('Rows by category:')
print(tariff_full_df['category'].value_counts(dropna=False))

tariff_full_df